In [ ]:
# Step 1 — Install libraries
!pip install transformers
!pip install datasets
!pip install sentencepiece
!pip install jiwer
!pip install ctranslate2
!pip install gradio
!pip install accelerate
!pip install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 21.0 MB/s eta 0:00:00


In [ ]:
# dowload dataset
!git clone https://huggingface.co/datasets/ai4bharat/Aksharantar

Cloning into 'Aksharantar'...
remote: Enumerating objects: 156, done.
remote: Total 156 (delta 0), reused 0 (delta 0), pack-reused 156 (from 1)
Receiving objects: 100% (156/156), 21.27 KiB | 10.63 MiB/s, done.
Resolving deltas: 100% (56/56), done.
Filtering content: 100% (21/21), 695.41 MiB | 61.21 MiB/s, done.


In [ ]:
import os
os.listdir("Aksharantar")

['README.md',
 '.git',
 'mni.zip',
 'kas.zip',
 'kan.zip',
 'hin.zip',
 'ori.zip',
 'tam.zip',
 'pan.zip',
 'mai.zip',
 'ben.zip',
 'brx.zip',
 'san.zip',
 'nep.zip',
 'urd.zip',
 'tel.zip',
 'guj.zip',
 'mar.zip',
 'sid.zip',
 'asm.zip',
 'mal.zip',
 '.gitattributes',
 'kok.zip',
 'doi.zip']

In [ ]:
!unzip Aksharantar/hin.zip
!unzip Aksharantar/ben.zip
!unzip Aksharantar/tam.zip

Archive:  Aksharantar/hin.zip
  inflating: hin_test.json           
  inflating: hin_train.json          
  inflating: hin_valid.json          
Archive:  Aksharantar/ben.zip
  inflating: ben_test.json           
  inflating: ben_train.json          
  inflating: ben_valid.json          
Archive:  Aksharantar/tam.zip
  inflating: tam_test.json           
  inflating: tam_train.json          
  inflating: tam_valid.json          


In [ ]:
import pandas as pd

hi = pd.read_json("/content/hin_train.json", lines=True)
bn = pd.read_json("/content/ben_train.json", lines=True)
ta = pd.read_json("/content/tam_train.json", lines=True)

In [ ]:
hi.head()

,unique_identifier,native word,english word,source,score
0,hin1,जन्मदिवस,janamdivas,Dakshina,NaN
1,hin2,रक्खा,rakha,Dakshina,NaN
2,hin3,मिलीजुली,milijuli,Dakshina,NaN
3,hin4,जांचों,jaanchon,Dakshina,NaN
4,hin5,चमकता,chamkata,Dakshina,NaN


In [ ]:
hi = hi[["english word", "native word"]].head(10000).copy()
bn = bn[["english word", "native word"]].head(10000).copy()
ta = ta[["english word", "native word"]].head(10000).copy()

In [ ]:
hi.head()

,english word,native word
0,janamdivas,जन्मदिवस
1,rakha,रक्खा
2,milijuli,मिलीजुली
3,jaanchon,जांचों
4,chamkata,चमकता


In [ ]:
def prepare(df, lang):
    return pd.DataFrame({
        "input": [f"transliterate to {lang}: {x}" for x in df["english word"].astype(str)],
        "target": [str(x) for x in df["native word"]]
    })

df_hi = prepare(hi, "hindi")
df_bn = prepare(bn, "bengali")
df_ta = prepare(ta, "tamil")

df = pd.concat([df_hi, df_bn, df_ta], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(df.head())
print(df.shape)

                                   input         target
0        transliterate to hindi: evready         एवरेडी
1       transliterate to tamil: selgirar     செல்கிறார்
2  transliterate to tamil: tholkaappiyam  தொல்காப்பியம்
3      transliterate to tamil: pesiyavar       பேசியவர்
4          transliterate to hindi: paaun           पाऊं
(30000, 2)


In [ ]:
df_hi = prepare(hi,"hindi")
df_bn = prepare(bn,"bengali")
df_ta = prepare(ta,"tamil")

In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df, preserve_index=False)

In [ ]:
dataset

Dataset({
    features: ['input', 'target'],
    num_rows: 30000
})

In [ ]:
dataset = dataset.train_test_split(test_size=0.1, seed=42)

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'target'],
        num_rows: 27000
    })
    test: Dataset({
        features: ['input', 'target'],
        num_rows: 3000
    })
})

In [ ]:
print(dataset["train"][0])

{'input': 'transliterate to tamil: aatchiyaalargalin', 'target': 'ஆட்சியாளர்களின்'}


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/byt5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/byt5-small")
model.config.use_cache = False

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
def tokenize(batch):
    model_inputs = tokenizer(
        batch["input"],
        truncation=True,
        max_length=32
    )

    labels = tokenizer(
        text_target=batch["target"],
        truncation=True,
        max_length=32
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["input", "target"]
)

print(tokenized_dataset)
print(tokenized_dataset["train"][0])

Map:   0%|          | 0/27000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 27000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
})
{'input_ids': [119, 117, 100, 113, 118, 111, 108, 119, 104, 117, 100, 119, 104, 35, 119, 114, 35, 119, 100, 112, 108, 111, 61, 35, 100, 100, 119, 102, 107, 108, 124, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [227, 177, 137, 227, 177, 162, 227, 178, 144, 227, 177, 157, 227, 177, 194, 227, 177, 178, 227, 177, 193, 227, 177, 182, 227, 177, 179, 227, 178, 144, 227, 1]}


In [ ]:
sample = tokenized_dataset["train"][0]
print("input_ids:", sample["input_ids"][:10])
print("labels:", sample["labels"][:10])
print("decoded input:", tokenizer.decode(sample["input_ids"], skip_special_tokens=True))
print("decoded label:", tokenizer.decode([x for x in sample["labels"] if x != -100], skip_special_tokens=True))

input_ids: [119, 117, 100, 113, 118, 111, 108, 119, 104, 117]
labels: [227, 177, 137, 227, 177, 162, 227, 178, 144, 227]
decoded input: transliterate to tamil: aatchiy
decoded label: ஆட்சியாளர்


In [ ]:
tokenized_dataset.save_to_disk("tokenized_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/27000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

In [ ]:
from datasets import load_from_disk
tokenized_dataset = load_from_disk("tokenized_dataset")

In [ ]:
sample = tokenized_dataset["train"][0]

print("INPUT IDS:", sample["input_ids"][:15])
print("LABEL IDS:", sample["labels"][:15])

print("DECODED INPUT:")
print(tokenizer.decode(sample["input_ids"], skip_special_tokens=True))

print("DECODED LABEL:")
print(tokenizer.decode(sample["labels"], skip_special_tokens=True))

INPUT IDS: [119, 117, 100, 113, 118, 111, 108, 119, 104, 117, 100, 119, 104, 35, 119]
LABEL IDS: [227, 177, 137, 227, 177, 162, 227, 178, 144, 227, 177, 157, 227, 177, 194]
DECODED INPUT:
transliterate to tamil: aatchiy
DECODED LABEL:
ஆட்சியாளர்


In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

training_args = Seq2SeqTrainingArguments(
    output_dir="./results_byt5",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    predict_with_generate=True,
    fp16=False,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)

In [ ]:
print(df.head(5))
print(df.isna().sum())
print(df["target"].head(10).tolist())
print(df["target"].str.len().describe())

                                   input         target
0        transliterate to hindi: evready         एवरेडी
1       transliterate to tamil: selgirar     செல்கிறார்
2  transliterate to tamil: tholkaappiyam  தொல்காப்பியம்
3      transliterate to tamil: pesiyavar       பேசியவர்
4          transliterate to hindi: paaun           पाऊं
input     0
target    0
dtype: int64
['एवरेडी', 'செல்கிறார்', 'தொல்காப்பியம்', 'பேசியவர்', 'पाऊं', 'कत्याल', 'अग्रसेन', 'क्रियाएं', 'সূর্যাস্তের', 'ঘটক']
count    30000.000000
mean         7.887500
std          2.896686
min          1.000000
25%          6.000000
50%          7.000000
75%         10.000000
max         25.000000
Name: target, dtype: float64


In [ ]:
print(dataset["train"][0])
print(dataset["train"][1])

{'input': 'transliterate to tamil: aatchiyaalargalin', 'target': 'ஆட்சியாளர்களின்'}
{'input': 'transliterate to bengali: seeksan', 'target': 'সেকশন'}


In [ ]:
print(tokenized_dataset["train"][0])

{'input_ids': [119, 117, 100, 113, 118, 111, 108, 119, 104, 117, 100, 119, 104, 35, 119, 114, 35, 119, 100, 112, 108, 111, 61, 35, 100, 100, 119, 102, 107, 108, 124, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [227, 177, 137, 227, 177, 162, 227, 178, 144, 227, 177, 157, 227, 177, 194, 227, 177, 178, 227, 177, 193, 227, 177, 182, 227, 177, 179, 227, 178, 144, 227, 1]}


In [ ]:
sample = tokenized_dataset["train"][0]

print("decoded input:")
print(tokenizer.decode(sample["input_ids"], skip_special_tokens=True))

print("raw labels:")
print(sample["labels"])

valid_labels = [x for x in sample["labels"] if x != -100]
print("valid label ids:")
print(valid_labels)

print("decoded label:")
print(tokenizer.decode(valid_labels, skip_special_tokens=True))

decoded input:
transliterate to tamil: aatchiy
raw labels:
[227, 177, 137, 227, 177, 162, 227, 178, 144, 227, 177, 157, 227, 177, 194, 227, 177, 178, 227, 177, 193, 227, 177, 182, 227, 177, 179, 227, 178, 144, 227, 1]
valid label ids:
[227, 177, 137, 227, 177, 162, 227, 178, 144, 227, 177, 157, 227, 177, 194, 227, 177, 178, 227, 177, 193, 227, 177, 182, 227, 177, 179, 227, 178, 144, 227, 1]
decoded label:
ஆட்சியாளர்


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.623391,0.495686
2,0.460778,0.412361
3,0.423510,0.396858


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5064, training_loss=0.7485408022128763, metrics={'train_runtime': 2201.2207, 'train_samples_per_second': 36.798, 'train_steps_per_second': 2.301, 'total_flos': 4651169734656000.0, 'train_loss': 0.7485408022128763, 'epoch': 3.0})

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

def transliterate(text, lang):
    prompt = f"transliterate to {lang}: {text}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=32)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            num_beams=4
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
print(transliterate("ram", "hindi"))
print(transliterate("ram", "bengali"))
print(transliterate("ram", "tamil"))
print(transliterate("namaste", "hindi"))
print(transliterate("krishna", "hindi"))

राम
রাম
ரம்
नमस्ते
क्रिश्


In [ ]:
from jiwer import cer,wer

refs=["राम","रम"]
preds=["राम","राम"]

print("CER:",cer(refs,preds))
print("WER:",wer(refs,preds))

CER: 0.2
WER: 0.5


In [ ]:
save_dir = "final_model"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('final_model/tokenizer_config.json', 'final_model/added_tokens.json')

In [ ]:
!ct2-transformers-converter \
  --model final_model \
  --output_dir ct2_model \
  --quantization int8

Loading weights: 100% 172/172 [00:00<00:00, 355.42it/s, Materializing param=shared.weight]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
import os
print(os.listdir(save_dir))

['generation_config.json', 'tokenizer_config.json', 'model.safetensors', 'config.json', 'training_args.bin', 'added_tokens.json']


In [ ]:
import os

def get_folder_size_mb(folder):
    total = 0
    for root, dirs, files in os.walk(folder):
        for file in files:
            path = os.path.join(root, file)
            total += os.path.getsize(path)
    return total / (1024 * 1024)

orig_size = get_folder_size_mb("final_model")
ct2_size = get_folder_size_mb("ct2_model")

print(f"Original model size: {orig_size:.2f} MB")
print(f"CTranslate2 model size: {ct2_size:.2f} MB")
print(f"Reduction: {((orig_size - ct2_size) / orig_size) * 100:.2f}%")

Original model size: 1147.40 MB
CTranslate2 model size: 287.21 MB
Reduction: 74.97%


In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

hf_tokenizer = AutoTokenizer.from_pretrained("final_model")
hf_model = AutoModelForSeq2SeqLM.from_pretrained("final_model").to(device)
hf_model.eval()

def hf_transliterate(text, lang):
    prompt = f"transliterate to {lang}: {text}"
    inputs = hf_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=64)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = hf_model.generate(**inputs, max_new_tokens=20, num_beams=4)
    return hf_tokenizer.decode(outputs[0], skip_special_tokens=True)

samples = [
    ("ram", "hindi"),
    ("namaste", "hindi"),
    ("krishna", "hindi"),
    ("bharat", "bengali"),
    ("aatchiyaalargalin", "tamil"),
] * 10

start = time.time()
for text, lang in samples:
    _ = hf_transliterate(text, lang)
hf_time = time.time() - start

print("HF total time:", hf_time)
print("HF avg latency per request:", hf_time / len(samples))

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


HF total time: 21.007375717163086
HF avg latency per request: 0.4201475143432617


In [ ]:
import time
import ctranslate2
from transformers import AutoTokenizer

ct2_tokenizer = AutoTokenizer.from_pretrained("final_model")
translator = ctranslate2.Translator("ct2_model", device="cpu")

def ct2_transliterate(text, lang):
    prompt = f"transliterate to {lang}: {text}"
    input_ids = ct2_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=64)["input_ids"][0]
    tokens = ct2_tokenizer.convert_ids_to_tokens(input_ids)
    results = translator.translate_batch([tokens], max_decoding_length=20, beam_size=4)
    output_tokens = results[0].hypotheses[0]
    return ct2_tokenizer.decode(ct2_tokenizer.convert_tokens_to_ids(output_tokens), skip_special_tokens=True)

samples = [
    ("ram", "hindi"),
    ("namaste", "hindi"),
    ("krishna", "hindi"),
    ("bharat", "bengali"),
    ("aatchiyaalargalin", "tamil"),
] * 10

start = time.time()
for text, lang in samples:
    _ = ct2_transliterate(text, lang)
ct2_time = time.time() - start

print("CT2 total time:", ct2_time)
print("CT2 avg latency per request:", ct2_time / len(samples))

CT2 total time: 46.610137701034546
CT2 avg latency per request: 0.932202754020691


In [ ]:
speed_gain = ((hf_time - ct2_time) / hf_time) * 100
print(f"Speed gain: {speed_gain:.2f}%")

Speed gain: -121.88%


In [ ]:
from jiwer import cer, wer

test_rows = df.sample(200, random_state=42)

preds = []
refs = []

for _, row in test_rows.iterrows():
    full_input = row["input"]
    target = row["target"]

    prefix = "transliterate to "
    rest = full_input[len(prefix):]
    lang, text = rest.split(": ", 1)

    pred = hf_transliterate(text, lang)

    preds.append(pred)
    refs.append(target)

exact_matches = sum(p == r for p, r in zip(preds, refs))
accuracy = exact_matches / len(refs)

print("Accuracy:", accuracy)
print("CER:", cer(refs, preds))
print("WER:", wer(refs, preds))

Accuracy: 0.09
CER: 0.40998104864181933
WER: 0.91


In [ ]:
ct2_preds = []

for _, row in test_rows.iterrows():
    full_input = row["input"]
    target = row["target"]

    prefix = "transliterate to "
    rest = full_input[len(prefix):]
    lang, text = rest.split(": ", 1)

    pred = ct2_transliterate(text, lang)
    ct2_preds.append(pred)

ct2_accuracy = sum(p == r for p, r in zip(ct2_preds, refs)) / len(refs)

print("CT2 Accuracy:", ct2_accuracy)
print("CT2 CER:", cer(refs, ct2_preds))
print("CT2 WER:", wer(refs, ct2_preds))

CT2 Accuracy: 0.075
CT2 CER: 0.4131396083385976
CT2 WER: 0.925


In [ ]:
# from huggingface_hub import notebook_login

# notebook_login()

In [ ]:
# model.push_to_hub("multilingual-transliteration")
# tokenizer.push_to_hub("multilingual-transliteration")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ypwt_lx/model.safetensors:   3%|3         | 39.9MB / 1.20GB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/kunalkurve219/multilingual-transliteration/commit/a7353afdb34d8e5ea00751afd25818c1dead122f', commit_message='Upload tokenizer', commit_description='', oid='a7353afdb34d8e5ea00751afd25818c1dead122f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kunalkurve219/multilingual-transliteration', endpoint='https://huggingface.co', repo_type='model', repo_id='kunalkurve219/multilingual-transliteration'), pr_revision=None, pr_num=None)

In [ ]:
# !zip -r final_model.zip final_model

  adding: final_model/ (stored 0%)
  adding: final_model/generation_config.json (deflated 29%)
  adding: final_model/tokenizer_config.json (deflated 95%)
  adding: final_model/model.safetensors (deflated 8%)
  adding: final_model/config.json (deflated 49%)
  adding: final_model/training_args.bin (deflated 53%)
  adding: final_model/added_tokens.json (deflated 82%)


In [ ]:
# from google.colab import files
# files.download("final_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# !zip -r ct2_model.zip ct2_model

  adding: ct2_model/ (stored 0%)
  adding: ct2_model/shared_vocabulary.json (deflated 82%)
  adding: ct2_model/model.bin (deflated 10%)
  adding: ct2_model/config.json (deflated 44%)


In [ ]:
# from google.colab import files
# files.download("ct2_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# tokenizer.save_pretrained("tokenizer_only")

('tokenizer_only/tokenizer_config.json', 'tokenizer_only/added_tokens.json')

In [ ]:
# !zip -r tokenizer_only.zip tokenizer_only

  adding: tokenizer_only/ (stored 0%)
  adding: tokenizer_only/tokenizer_config.json (deflated 95%)
  adding: tokenizer_only/added_tokens.json (deflated 82%)
